# Experiment D — RQ3 mixed populations + Traitor injection  
## ΠΡΟΣΟΧΗ: template — ενεργοποιείται μετά το AgentSpec refactor

**Τι θα ελέγξουμε (D1, RQ3):** όταν ο hub και τα leaves είναι ΔΙΑΦΟΡΕΤΙΚΑ μοντέλα με γνωστά καθεστώτα (Qwen3-4B ritual, Gemma-2-9B relevance-gated), ποιανού το καθεστώς κληρονομεί ο πληθυσμός; Mixes: (i) Qwen3 hub + Gemma-9B leaves, (ii) το αντίστροφο. Scenarios: no_comm, baseline, framing_competitive. PD, N=5.

**Τι θα ελέγξουμε (D2, traitor):** ένας deceptive agent (system_suffix: χτίσε εμπιστοσύνη με τα μηνύματα, μεγιστοποίησε το payoff σου με τις πράξεις) μέσα σε faithful Qwen2.5 πληθυσμό. Δύο θέσεις: hub-traitor και leaf-traitor. Μετρικές: TAS, DES, χρόνος προσαρμογής.

**Γιατί μετά το refactor:** ο ModelConfig είναι σήμερα ένας για όλους τους agents. Το `AgentSpec` (per-agent model + role + system_suffix, ένα μοντέλο ανά T4 GPU) είναι το επόμενο προγραμματισμένο βήμα κώδικα.

## Setup — install, GPU check, clone, HF token

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN` (χρειάζεται για Llama/Gemma)

**Προσοχή:** το repo πρέπει να έχει γίνει push με τις αλλαγές του Phase 1.5 (topologies + `--action-retries`) πριν τρέξει αυτό το notebook.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU not enabled!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
import os
if not os.path.exists('/kaggle/working/repo'):
    !git clone $GITHUB_REPO /kaggle/working/repo
%cd /kaggle/working/repo
# sanity: Phase-1.5 features present
assert 'clique' in open('topology.py').read(), 'Repo lacks Phase-1.5 topologies — push first!'
!ls

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded')
except Exception as e:
    print('No HF_TOKEN secret (fine for Qwen):', e)

In [ ]:
# Gate: αποτυγχάνει με σαφές μήνυμα αν το refactor δεν έχει μπει ακόμα
import importlib, config
importlib.reload(config)
assert hasattr(config, 'AgentSpec'), (
    'AgentSpec refactor not in the repo yet — αυτό το notebook ενεργοποιείται στο επόμενο βήμα.')

In [ ]:
# Σχεδιασμένες εντολές (θα οριστικοποιηθούν με το refactor):
# D1: !python run_mixed.py --hub Qwen/Qwen3-4B --leaves google/gemma-2-9b-it \
#         --scenarios baseline framing_competitive --n-runs 5
# D1': ίδιο με --hub google/gemma-2-9b-it --leaves Qwen/Qwen3-4B
# D2: !python run_traitor.py --model Qwen/Qwen2.5-7B-Instruct --traitor-position hub --n-runs 5
# D2': ίδιο με --traitor-position leaf